# Clase 11 · Notebook 02
# Optimización de hiperparámetros con búsqueda aleatoria

**Introducción al Deep Learning — Módulo 2, Unidad 2: Clasificación (Parte II)**

Vamos a **afinar** un clasificador buscando la mejor combinación de hiperparámetros. Seguiremos las cuatro
fases vistas en teoría con una **búsqueda aleatoria** (Random Search), implementada a mano para entender
el proceso.

1. Definir el espacio de hiperparámetros.
2. Elegir el algoritmo de búsqueda (aleatoria).
3. Buscar: probar configuraciones y evaluarlas.
4. Seleccionar la mejor y evaluarla en el test.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Datos y conjuntos (entrenamiento / validación / test)

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
tf.random.set_seed(42); np.random.seed(42)

data = load_breast_cancer()
X = StandardScaler().fit_transform(data.data); y = data.target
# train / val / test
Xtmp, Xte, ytmp, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
Xtr, Xval, ytr, yval = train_test_split(Xtmp, ytmp, test_size=0.2, random_state=42, stratify=ytmp)
print("train:", Xtr.shape[0], "| val:", Xval.shape[0], "| test:", Xte.shape[0])

## 2. Espacio de hiperparámetros y algoritmo de búsqueda

Definimos los valores posibles de cada hiperparámetro. La **búsqueda aleatoria** toma combinaciones al azar
de este espacio (más eficiente que probarlas todas).

In [ ]:
import random
random.seed(42)

espacio = {
    "neuronas":      [8, 16, 32, 64],
    "activacion":    ["relu", "tanh"],
    "learning_rate": [0.01, 0.005, 0.001],
    "batch":         [16, 32],
}
print("Espacio de búsqueda:")
for k, v in espacio.items():
    print(f"  {k}: {v}")

## 3. Buscar: entrenar y evaluar cada configuración

Para cada configuración aleatoria construimos la red, la entrenamos y medimos su **accuracy de validación**
(nunca el test, que se reserva para el final).

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

def construir_y_evaluar(cfg):
    m = Sequential([
        Input(shape=(X.shape[1],)),
        Dense(cfg["neuronas"], activation=cfg["activacion"]),
        Dense(cfg["neuronas"], activation=cfg["activacion"]),
        Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(cfg["learning_rate"]),
              loss="binary_crossentropy", metrics=["accuracy"])
    m.fit(Xtr, ytr, epochs=30, batch_size=cfg["batch"], verbose=0)
    val_acc = m.evaluate(Xval, yval, verbose=0)[1]
    return m, val_acc

N_INTENTOS = 6
resultados = []
for i in range(N_INTENTOS):
    cfg = {k: random.choice(v) for k, v in espacio.items()}
    modelo, val_acc = construir_y_evaluar(cfg)
    resultados.append((val_acc, cfg, modelo))
    print(f"intento {i+1}: val_acc = {val_acc:.3f}  <- {cfg}")

## 4. Seleccionar la mejor configuración y evaluarla en el test

In [ ]:
resultados.sort(key=lambda r: r[0], reverse=True)
mejor_val, mejor_cfg, mejor_modelo = resultados[0]

print("Mejor configuración encontrada:")
for k, v in mejor_cfg.items():
    print(f"  {k}: {v}")
print(f"\nAccuracy de validación: {mejor_val:.3f}")
test_acc = mejor_modelo.evaluate(Xte, yte, verbose=0)[1]
print(f"Accuracy en TEST (evaluación final): {test_acc:.3f}")

La búsqueda aleatoria nos ha dado, con pocas pruebas, una buena configuración. La evaluación final se
hace **una sola vez** sobre el test, que no se usó durante la búsqueda.

## Experimenta tú

1. Aumenta `N_INTENTOS` y observa si mejora la mejor configuración.
2. Amplía el espacio (más valores de `neuronas` o `learning_rate`).
3. Compara con una **búsqueda en cuadrícula** (probar TODAS las combinaciones) y mide cuánto tarda.

## En la práctica: Keras Tuner

Frameworks como **Keras Tuner** automatizan todo esto (random search, grid, bayesiana):

```python
import keras_tuner as kt
def build(hp):
    m = Sequential()
    m.add(Dense(hp.Int("neuronas", 8, 64, step=8), activation="relu"))
    m.add(Dense(1, activation="sigmoid"))
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return m
tuner = kt.RandomSearch(build, objective="val_accuracy", max_trials=10)
tuner.search(Xtr, ytr, validation_data=(Xval, yval), epochs=30)
mejor = tuner.get_best_models(1)[0]
```

## Resumen

- La **optimización de hiperparámetros** busca la mejor configuración de entrenamiento.
- **Búsqueda aleatoria**: probar combinaciones al azar; eficiente y sencilla.
- Proceso en 4 fases: definir → elegir algoritmo → buscar (con validación) → evaluar la mejor en test.
- Herramientas como **Keras Tuner** lo automatizan.

Con esto cerramos la segunda parte de la unidad de clasificación.